# 01 Cleaning: seller late handover

This notebook turns the raw Olist CSVs into the cleaned order and seller tables used by every later notebook. Each step prints the rows it removes so the counts can be quoted in the report. The decisions behind each step are recorded in `notebooks/change.markdown`.

| Output (`Data/Clean/`) | Contents |
| --- | --- |
| `orders_main.csv` | Main sample: delivered, single-seller orders with valid timestamps, January 2017 to August 2018 |
| `orders_group_b.csv` | Orders handed over after purchase but before payment approval; `04_checks` only |
| `seller_metrics.csv` | One row per seller, built from `orders_main.csv` |
| `monthly_benchmark.csv` | Platform late rate by purchase month |

Run from top to bottom from the `notebooks/` folder.

### Load the Olist data

Only the order and item tables are needed to measure late handover. Load them from the project's GitHub repository (`Data/Raw`).

In [1]:
from pathlib import Path

import pandas as pd

data_url = (
    "https://raw.githubusercontent.com/"
    "RitaMa1999/Python-Group-Project---ECommerce/main/Data/Raw"
)
output_dir = Path("../Data/Clean")
output_dir.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(f"{data_url}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{data_url}/olist_order_items_dataset.csv")

print(f"orders: {len(orders):,} rows")
print(f"order_items: {len(order_items):,} rows")

orders: 99,441 rows
order_items: 112,650 rows


### Validate the order key

Blank or duplicate order IDs would make an order-level merge unreliable.

In [2]:
order_ids = orders["order_id"]
missing_ids = order_ids.isna() | order_ids.astype("string").str.strip().eq("").fillna(False)
duplicate_ids = order_ids.duplicated(keep=False) & ~missing_ids

print(f"Rows with missing or blank order_id: {missing_ids.sum()}")
print(f"Rows with duplicate order_id: {duplicate_ids.sum()}")

if (missing_ids | duplicate_ids).any():
    raise ValueError("Review missing or duplicate order IDs before merging.")

Rows with missing or blank order_id: 0
Rows with duplicate order_id: 0


### Parse timestamps and standardize status

Convert the four order timestamps and the item shipping deadline to datetimes, report parse failures, and normalize status labels.

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

for column in date_columns:
    parsed = pd.to_datetime(orders[column], errors="coerce")
    failed = orders[column].notna() & parsed.isna()
    print(f"{column}: {failed.sum()} values could not be parsed")
    orders[column] = parsed

parsed_deadline = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
failed = order_items["shipping_limit_date"].notna() & parsed_deadline.isna()
print(f"shipping_limit_date: {failed.sum()} values could not be parsed")
order_items["shipping_limit_date"] = parsed_deadline

orders["order_status"] = orders["order_status"].astype("string").str.strip().str.lower()
display(orders["order_status"].value_counts().rename("orders").to_frame())

order_purchase_timestamp: 0 values could not be parsed
order_approved_at: 0 values could not be parsed
order_delivered_carrier_date: 0 values could not be parsed
order_delivered_customer_date: 0 values could not be parsed
shipping_limit_date: 0 values could not be parsed


,orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## Data audit

These checks show the problems found in the raw data. The cleaning steps below act on them.

### Missing handover timestamps and order status (change 1)

Almost all orders without a handover timestamp were never delivered. Shipped and canceled orders that do have a handover timestamp have no final delivered outcome.

In [4]:
missing_handover = orders["order_delivered_carrier_date"].isna()
print(f"Orders missing a handover timestamp: {missing_handover.sum():,} "
      f"({missing_handover.mean():.2%})")

display(
    pd.crosstab(
        orders["order_status"],
        orders["order_delivered_carrier_date"].notna().rename("has_handover_timestamp"),
    )
)

Orders missing a handover timestamp: 1,783 (1.79%)


has_handover_timestamp,False,True
order_status,,
approved,2,0
canceled,550,75
created,5,0
delivered,2,96476
invoiced,314,0
processing,301,0
shipped,0,1107
unavailable,609,0


### Timestamps in the wrong order (change 2)

Compare each pair of timestamps across all orders. The cleaning step repeats this inside the delivered, single-seller sample.

In [5]:
purchase, approved, handover, customer = (orders[column] for column in date_columns)

display(
    pd.Series({
        "Handover before purchase": handover.lt(purchase).sum(),
        "Handover before payment approval": handover.lt(approved).sum(),
        "Customer delivery before handover": customer.lt(handover).sum(),
    }, name="orders").to_frame()
)

,orders
Handover before purchase,166
Handover before payment approval,1359
Customer delivery before handover,23


### Shipping deadlines years after purchase (change 3)

Deadlines are normally a few days after purchase. Four item rows from one seller fall in 2020.

In [6]:
item_deadlines = order_items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="many_to_one",
)
item_deadlines["days_to_deadline"] = (
    item_deadlines["shipping_limit_date"]
    .sub(item_deadlines["order_purchase_timestamp"])
    .dt.total_seconds()
    .div(86400)
)

display(item_deadlines["shipping_limit_date"].dt.year.value_counts().sort_index().rename("item_rows").to_frame())
display(item_deadlines["days_to_deadline"].describe(percentiles=[0.5, 0.99]).round(1).to_frame())

far_deadline = item_deadlines["days_to_deadline"].gt(365)
display(item_deadlines.loc[far_deadline, ["order_id", "seller_id", "shipping_limit_date", "days_to_deadline"]])

long_deadline = item_deadlines["days_to_deadline"].between(30, 365)
print(f"Item rows with a deadline 30-365 days after purchase (kept): {long_deadline.sum():,}")

,item_rows
shipping_limit_date,
2016,370
2017,49765
2018,62511
2020,4


,days_to_deadline
count,112650.0
mean,6.6
std,7.1
min,2.0
50%,6.0
99%,19.0
max,1056.0


,order_id,seller_id,shipping_limit_date,days_to_deadline
8643,13bdf405f961a6deec817d817f5c6624,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,1056.041667
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,1056.041667
85729,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,1052.004537
85730,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,1052.004537


Item rows with a deadline 30-365 days after purchase (kept): 235


### Orders with no items and orders with more than one seller (change 5)

`orders` records one handover timestamp per order, and `order_items` has none, so a multi-seller order's handover cannot be attributed to one seller.

In [7]:
has_items = orders["order_id"].isin(order_items["order_id"])
print(f"Orders with no item rows: {(~has_items).sum():,}")
display(orders.loc[~has_items, "order_status"].value_counts().rename("orders").to_frame())

sellers_per_order = order_items.groupby("order_id")["seller_id"].nunique()
display(sellers_per_order.value_counts().sort_index().rename_axis("sellers_in_order").rename("orders").to_frame())

Orders with no item rows: 775


,orders
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


,orders
sellers_in_order,
1,97388
2,1219
3,54
4,3
5,2


### Edge months (change 4)

The first and last months of the data are very thin. September and October 2018 have no delivered orders.

In [8]:
purchase_month = orders["order_purchase_timestamp"].dt.to_period("M")
edge_months = (
    purchase_month.dt.year.eq(2016)
    | purchase_month.isin(pd.PeriodIndex(["2018-09", "2018-10"], freq="M"))
)
display(pd.crosstab(purchase_month[edge_months], orders.loc[edge_months, "order_status"]))

order_status,canceled,delivered,invoiced,processing,shipped,unavailable
order_purchase_timestamp,,,,,,
2016-09,2,1,0,0,1,0
2016-10,24,265,18,2,8,7
2016-12,0,1,0,0,0,0
2018-09,15,0,0,0,1,0
2018-10,4,0,0,0,0,0


## Cleaning

### Build one row per order and seller

A seller can supply several items in one order. Check for conflicting deadlines, then keep one row per order and seller. Orders with no items never enter this table.

In [9]:
for column in ["order_id", "seller_id"]:
    order_items[column] = order_items[column].astype("string").str.strip().replace("", pd.NA)

if order_items[["order_id", "seller_id"]].isna().any().any():
    raise ValueError("Review item rows with missing IDs before grouping.")

pair_deadlines = order_items.groupby(["order_id", "seller_id"])["shipping_limit_date"]
conflicting_pairs = pair_deadlines.nunique().gt(1)
if conflicting_pairs.any():
    raise ValueError("Review conflicting shipping deadlines before grouping.")

seller_orders = pair_deadlines.first().reset_index()
seller_orders["seller_count"] = seller_orders.groupby("order_id")["seller_id"].transform("nunique")

print(f"Item rows: {len(order_items):,}")
print(f"Order-seller rows: {len(seller_orders):,}")
print(f"Multi-seller orders: {seller_orders.loc[seller_orders['seller_count'].gt(1), 'order_id'].nunique():,}")

Item rows: 112,650
Order-seller rows: 100,010
Multi-seller orders: 1,278


### Keep single-seller orders and attach order details (change 5)

A late handover in a multi-seller order cannot be attributed to one seller.

In [10]:
single_seller = seller_orders.loc[seller_orders["seller_count"].eq(1)].drop(columns="seller_count")

clean_orders = single_seller.merge(
    orders,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

if clean_orders["_merge"].ne("both").any():
    raise ValueError("Some single-seller orders have no matching order record.")
clean_orders = clean_orders.drop(columns="_merge")

print(f"Single-seller orders: {len(clean_orders):,}")

Single-seller orders: 97,388


### Keep delivered orders (change 1)

Orders that were not delivered have no final handover outcome. This removes shipped orders, canceled orders and almost all orders missing a handover timestamp.

In [11]:
rows_before = len(clean_orders)
clean_orders = clean_orders.loc[clean_orders["order_status"].eq("delivered")].copy()

print(f"Removed non-delivered orders: {rows_before - len(clean_orders):,}")
print(f"Delivered, single-seller orders: {len(clean_orders):,}")

Removed non-delivered orders: 2,185
Delivered, single-seller orders: 95,203


### Check timestamp completeness and order (change 2)

Keep orders whose four timestamps are present and in the order purchase, approval, handover, customer delivery. Group B (handed over after purchase but before approval) may reflect late approval records, so it is saved separately for `04_checks`.

In [12]:
missing_timestamp = clean_orders[date_columns].isna().any(axis=1)
print("Missing timestamps among delivered, single-seller orders:")
display(clean_orders[date_columns].isna().sum().rename("missing").to_frame())

clean_orders = clean_orders.loc[~missing_timestamp].copy()
print(f"Removed for a missing timestamp: {missing_timestamp.sum():,}")

purchase, approved, handover, customer = (clean_orders[column] for column in date_columns)
group_a = handover.lt(purchase)
group_b = handover.ge(purchase) & handover.lt(approved) & customer.ge(handover)
customer_before_handover = customer.lt(handover)
in_order = purchase.le(approved) & approved.le(handover) & handover.le(customer)

display(
    pd.Series({
        "Group A: handover before purchase": group_a.sum(),
        "Group B: handover after purchase, before approval": group_b.sum(),
        "Customer delivery before handover": customer_before_handover.sum(),
        "Timestamps in order": in_order.sum(),
    }, name="orders").to_frame()
)

gap_hours = approved.sub(handover)[group_b].dt.total_seconds().div(3600)
print("Group B: hours between handover and approval")
display(gap_hours.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame())

group_b_orders = clean_orders.loc[group_b].copy()
clean_orders = clean_orders.loc[in_order].copy()
print(f"Orders with timestamps in order: {len(clean_orders):,}")

Missing timestamps among delivered, single-seller orders:


,missing
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,2
order_delivered_customer_date,8


Removed for a missing timestamp: 23


,orders
Group A: handover before purchase,161
"Group B: handover after purchase, before approval",1156
Customer delivery before handover,4
Timestamps in order,93859


Group B: hours between handover and approval


,0
count,1156.0
mean,24.1
std,32.4
min,0.0
50%,19.8
90%,54.9
99%,165.6
max,224.9


Orders with timestamps in order: 93,859


### Drop shipping deadlines more than a year after purchase (change 3)

These deadlines are entry errors. Drop the whole order rather than replacing the deadline.

In [13]:
far_order_ids = item_deadlines.loc[far_deadline, "order_id"].unique()

removed_main = clean_orders["order_id"].isin(far_order_ids)
removed_group_b = group_b_orders["order_id"].isin(far_order_ids)
clean_orders = clean_orders.loc[~removed_main].copy()
group_b_orders = group_b_orders.loc[~removed_group_b].copy()

print(f"Orders with a deadline more than 365 days after purchase: {len(far_order_ids)}")
print(f"Removed from the main sample: {removed_main.sum()}; from group B: {removed_group_b.sum()}")

Orders with a deadline more than 365 days after purchase: 3
Removed from the main sample: 1; from group B: 0


### Restrict the study period to January 2017 – August 2018 (change 4)

2016 was the platform's launch period: few orders, a gap in November and platform-wide slow handovers.

In [14]:
is_2016 = clean_orders["order_purchase_timestamp"].dt.year.eq(2016)

days_after_approval = lambda frame, column: (
    frame[column].sub(frame["order_approved_at"]).dt.total_seconds().div(86400)
)
display(
    pd.DataFrame({
        "orders": is_2016.value_counts(),
        "median_days_approval_to_deadline": days_after_approval(clean_orders, "shipping_limit_date").groupby(is_2016).median(),
        "median_days_approval_to_handover": days_after_approval(clean_orders, "order_delivered_carrier_date").groupby(is_2016).median(),
    })
    .rename(index={True: "2016", False: "2017-2018"})
    .round(1)
)

sellers_2016 = set(clean_orders.loc[is_2016, "seller_id"])
sellers_later = set(clean_orders.loc[~is_2016, "seller_id"])
print(f"Sellers with 2016 orders: {len(sellers_2016)}; only in 2016: {len(sellers_2016 - sellers_later)}")

group_b_2016 = group_b_orders["order_purchase_timestamp"].dt.year.eq(2016)
clean_orders = clean_orders.loc[~is_2016].copy()
group_b_orders = group_b_orders.loc[~group_b_2016].copy()

print(f"Removed 2016 orders: {is_2016.sum()} from the main sample, {group_b_2016.sum()} from group B")
print(f"Main sample: {len(clean_orders):,} orders; group B: {len(group_b_orders):,} orders")

,orders,median_days_approval_to_deadline,median_days_approval_to_handover
order_purchase_timestamp,,,
2017-2018,93598,6.0,1.8
2016,260,4.0,8.9


Sellers with 2016 orders: 126; only in 2016: 23
Removed 2016 orders: 260 from the main sample, 0 from group B
Main sample: 93,598 orders; group B: 1,156 orders


### Define late handover by date (change 6)

The deadline's clock time is the payment approval time plus whole days, so it often falls in the early morning while carriers collect in the afternoon. An order is late when it is handed over on a calendar day after the deadline day.

In [15]:
def add_late_handover(frame):
    frame = frame.copy()
    frame["late_handover"] = (
        frame["order_delivered_carrier_date"].dt.normalize()
        .gt(frame["shipping_limit_date"].dt.normalize())
    )
    frame["order_purchase_month"] = frame["order_purchase_timestamp"].dt.to_period("M").astype(str)
    return frame

clean_orders = add_late_handover(clean_orders)
group_b_orders = add_late_handover(group_b_orders)

handover = clean_orders["order_delivered_carrier_date"]
deadline = clean_orders["shipping_limit_date"]
approved = clean_orders["order_approved_at"]
late_by_timestamp = handover.gt(deadline)
late_by_date = clean_orders["late_handover"]

print(f"Late by date (used): {late_by_date.sum():,} ({late_by_date.mean():.2%})")
print(f"Late by timestamp (04_checks): {late_by_timestamp.sum():,} ({late_by_timestamp.mean():.2%})")
print(f"Late by timestamp only, handed over on the deadline day: {(late_by_timestamp & ~late_by_date).sum():,}")

whole_days = deadline.sub(approved).dt.total_seconds().mod(86400).eq(0)
same_clock_time = deadline.dt.strftime("%H:%M").eq(approved.dt.strftime("%H:%M"))
print(f"Deadline = approval + whole days: {whole_days.mean():.0%}")
print(f"Deadline clock time = approval clock time: {same_clock_time.mean():.0%}")

display(
    pd.DataFrame({
        "deadline_share": deadline.dt.hour.value_counts(normalize=True),
        "handover_share": handover.dt.hour.value_counts(normalize=True),
    })
    .sort_index()
    .rename_axis("hour")
    .round(3)
    .T
)

Late by date (used): 5,407 (5.78%)
Late by timestamp (04_checks): 8,522 (9.10%)
Late by timestamp only, handed over on the deadline day: 3,115


Deadline = approval + whole days: 80%
Deadline clock time = approval clock time: 82%


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
deadline_share,0.022,0.014,0.073,0.057,0.035,0.009,0.009,0.014,0.023,0.041,...,0.054,0.054,0.055,0.055,0.048,0.049,0.051,0.049,0.049,0.036
handover_share,0.025,0.014,0.006,0.003,0.002,0.003,0.007,0.011,0.020,0.026,...,0.107,0.090,0.059,0.058,0.071,0.079,0.070,0.058,0.049,0.038


### Save the cleaned orders

`orders_main.csv` is the main sample. `orders_group_b.csv` has the same columns and is only for `04_checks`.

In [16]:
order_columns = [
    "order_id",
    "seller_id",
    "customer_id",
    "order_status",
    *date_columns,
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "order_purchase_month",
    "late_handover",
]
orders_main = clean_orders[order_columns].sort_values("order_purchase_timestamp").reset_index(drop=True)
orders_group_b = group_b_orders[order_columns].sort_values("order_purchase_timestamp").reset_index(drop=True)

orders_main.to_csv(output_dir / "orders_main.csv", index=False)
orders_group_b.to_csv(output_dir / "orders_group_b.csv", index=False)

print(f"orders_main.csv: {len(orders_main):,} orders, "
      f"{orders_main['order_purchase_month'].min()} to {orders_main['order_purchase_month'].max()}")
print(f"orders_group_b.csv: {len(orders_group_b):,} orders")

orders_main.csv: 93,598 orders, 2017-01 to 2018-08
orders_group_b.csv: 1,156 orders


### Build the monthly benchmark and seller metrics (change 7)

`benchmark_late_rate` is the late rate a seller would have if each of its orders were late at the platform rate for that month. `late_rate_gap_pp` is the seller's late rate minus that benchmark, in percentage points. No minimum-order threshold is applied here; the clustering team chooses it.

In [17]:
monthly_benchmark = (
    orders_main.groupby("order_purchase_month", as_index=False)
    .agg(orders=("order_id", "size"), late_orders=("late_handover", "sum"))
)
monthly_benchmark["late_rate"] = monthly_benchmark["late_orders"] / monthly_benchmark["orders"]

orders_with_benchmark = orders_main.merge(
    monthly_benchmark[["order_purchase_month", "late_rate"]].rename(columns={"late_rate": "month_late_rate"}),
    on="order_purchase_month",
    how="left",
    validate="many_to_one",
)
seller_metrics = (
    orders_with_benchmark.groupby("seller_id", as_index=False)
    .agg(
        order_volume=("order_id", "size"),
        late_orders=("late_handover", "sum"),
        benchmark_late_rate=("month_late_rate", "mean"),
    )
)
seller_metrics["late_rate"] = seller_metrics["late_orders"] / seller_metrics["order_volume"]
seller_metrics["late_rate_gap_pp"] = (
    seller_metrics["late_rate"].sub(seller_metrics["benchmark_late_rate"]).mul(100)
)
seller_metrics = seller_metrics[
    ["seller_id", "order_volume", "late_orders", "late_rate", "benchmark_late_rate", "late_rate_gap_pp"]
]

monthly_benchmark.to_csv(output_dir / "monthly_benchmark.csv", index=False)
seller_metrics.to_csv(output_dir / "seller_metrics.csv", index=False)

display(monthly_benchmark.round(4))
display(seller_metrics.describe().round(3))

,order_purchase_month,orders,late_orders,late_rate
0,2017-01,744,138,0.1855
1,2017-02,1628,247,0.1517
2,2017-03,2541,136,0.0535
3,2017-04,2261,121,0.0535
4,2017-05,3498,182,0.0520
5,2017-06,3106,165,0.0531
6,2017-07,3788,137,0.0362
7,2017-08,4134,152,0.0368
8,2017-09,4093,164,0.0401
9,2017-10,4414,199,0.0451


,order_volume,late_orders,late_rate,benchmark_late_rate,late_rate_gap_pp
count,2917.000,2917.000,2917.000,2917.000,2917.000
mean,32.087,1.854,0.086,0.059,2.659
std,102.396,7.273,0.207,0.020,20.425
min,1.000,0.000,0.000,0.035,-18.548
25%,2.000,0.000,0.000,0.050,-6.014
50%,7.000,0.000,0.000,0.056,-4.796
75%,22.000,1.000,0.061,0.062,0.189
max,1737.000,189.000,1.000,0.185,96.507


### Reconcile the outputs

Check that the saved files agree with each other and reload correctly.

In [18]:
reloaded_orders = pd.read_csv(output_dir / "orders_main.csv")
reloaded_sellers = pd.read_csv(output_dir / "seller_metrics.csv")
reloaded_months = pd.read_csv(output_dir / "monthly_benchmark.csv")

platform_late_rate = orders_main["late_handover"].mean()
weighted_benchmark = (
    reloaded_sellers["benchmark_late_rate"].mul(reloaded_sellers["order_volume"]).sum()
    / reloaded_sellers["order_volume"].sum()
)

checks = pd.DataFrame({
    "value": [
        len(reloaded_orders),
        reloaded_orders["order_id"].nunique(),
        reloaded_sellers["order_volume"].sum(),
        reloaded_months["orders"].sum(),
        reloaded_orders["late_handover"].sum(),
        reloaded_sellers["late_orders"].sum(),
        reloaded_months["late_orders"].sum(),
    ],
}, index=[
    "Orders in orders_main.csv",
    "Distinct order IDs",
    "Orders across sellers",
    "Orders across months",
    "Late orders in orders_main.csv",
    "Late orders across sellers",
    "Late orders across months",
])
display(checks)

if checks.loc[["Distinct order IDs", "Orders across sellers", "Orders across months"], "value"].ne(len(reloaded_orders)).any():
    raise ValueError("Order counts do not reconcile.")
if checks.loc[["Late orders across sellers", "Late orders across months"], "value"].ne(reloaded_orders["late_handover"].sum()).any():
    raise ValueError("Late order counts do not reconcile.")
if abs(weighted_benchmark - platform_late_rate) > 1e-9:
    raise ValueError("Weighted seller benchmark does not equal the platform late rate.")

print(f"Platform late rate: {platform_late_rate:.2%}")
print(f"Sellers: {len(reloaded_sellers):,}; months: {len(reloaded_months)}")

,value
Orders in orders_main.csv,93598
Distinct order IDs,93598
Orders across sellers,93598
Orders across months,93598
Late orders in orders_main.csv,5407
Late orders across sellers,5407
Late orders across months,5407


Platform late rate: 5.78%
Sellers: 2,917; months: 20


## Handoff to the clustering team

Run this notebook from top to bottom from the `notebooks/` folder. It loads the raw CSVs from the project's GitHub repository (`Data/Raw`) and writes the files below to `Data/Clean/`.

| File | What to use it for |
| --- | --- |
| `Data/Clean/seller_metrics.csv` | One row per seller (2,917): `order_volume`, `late_orders`, `late_rate`, `benchmark_late_rate`, `late_rate_gap_pp`. Input for Q1, Q2 clustering and figures. No minimum-order threshold applied. |
| `Data/Clean/monthly_benchmark.csv` | Platform late rate by purchase month, 2017-01 to 2018-08. Same-month baseline for Q1 and the monthly trend figure. |
| `Data/Clean/orders_main.csv` | Main sample, one row per order (93,598). Use it to rebuild seller metrics under other settings, such as the 2017 training and 2018 validation split. |
| `Data/Clean/orders_group_b.csv` | 1,156 orders handed over after purchase but before payment approval. Use only in `04_checks`: combine with `orders_main.csv` and compare results. |

All main analysis (Q1, Q2, figures, report and slide numbers) reads `seller_metrics.csv` and `monthly_benchmark.csv`. Do not add `orders_group_b.csv` to the main analysis.

Late handover is defined by date: an order is late when it is handed to the carrier on a calendar day after the `shipping_limit_date` day. `late_rate` is a proportion from 0 to 1. The platform late rate in the main sample is 5.78%.

### Minimum orders per seller (clustering team decides)

Many sellers have very few orders, so their late rates are unreliable. In the main sample, 79% of sellers with fewer than 5 orders have a 0% late rate and 10% have 100%.

| Minimum orders | Sellers kept | Share of sellers | Share of orders |
| --- | --- | --- | --- |
| None | 2,917 | 100% | 100% |
| 5 | 1,715 | 58.8% | 97.4% |
| 10 | 1,199 | 41.1% | 93.7% |
| 20 | 778 | 26.7% | 87.5% |
| 30 | 604 | 20.7% | 83.0% |

- The group plan suggests 20 orders for the main analysis, with 10 and 30 as checks in `04_checks`.
- Apply the threshold to sellers in the clustering step only. The platform late rate, the monthly benchmark and the Q1 distributions use all orders.
- Explain the chosen threshold in the report and show how results change under the alternatives.

### 2017 → 2018 holdout

Few sellers are active enough in both years. Rebuild yearly seller metrics from `orders_main.csv` and check this count early.

| Minimum orders in each year | Sellers in 2017 | Sellers in 2018 | Sellers in both years |
| --- | --- | --- | --- |
| 10 | 640 | 876 | 429 |
| 20 | 420 | 527 | 264 |
| 30 | 309 | 368 | 171 |

### Credibility checks for `04_checks`

**Handover before payment approval.** The main sample drops 1,156 orders that were handed over after purchase but before payment approval (saved separately as `Data/Clean/orders_group_b.csv`). These may be approval delays rather than data errors. In `04_checks`, combine the two files, rerun the seller metrics and K-means, then compare the platform late rate, seller late rates, k, cluster sizes and profiles, and the number of sellers that change cluster. If the results are similar, state in the report that this filter does not affect the conclusions. The 161 orders handed over before purchase stay dropped in both versions.

**Late handover definition.** The main analysis defines late handover by date (handed over on a day after the deadline day). In `04_checks`, rerun the seller metrics and K-means with the timestamp definition (`order_delivered_carrier_date > shipping_limit_date`), then compare the platform late rate (9.10% by timestamp), seller late rates, cluster profiles and the sellers in the high-priority group.

## Appendix A — Use of Generative AI

I used ChatGPT to draft and revise pandas code for the first version of this notebook (A.1–A.4), and Claude (Anthropic) to review that version against the raw data and rebuild it (A.5). The prompts below document how the notebook developed in stages.

### A.1 Order data validation and cleaning

> Please write a python code to check the values in `order_id` column in the `orders` dataset are unique and complete, so the existing records can be retained.

> write a python code to standardize the time format. Convert the order placement time (`order_purchase_timestamp`) and the actual delivery time (`order_delivered_carrier_date`) to a date-time format. Both them in the order dataset. The order placement time will later be used for month-to-month comparisons.

> please write a code to check the Missing and Abnormal Values for three columns: **order_purchase_timestamp**, **order_delivered_carrier_date**, **order_purchase_month**

> Write the code to clean the `order_status` column in my existing `orders` DataFrame using pandas. The previous already check order IDs, parse purchase and carrier handover timestamps, and flag timestamp issues. Continue from the current `orders` DataFrame.

> Then Convert `order_status` to pandas’ nullable string type, strip leading and trailing whitespace, convert values to lowercase, and treat empty strings as missing. Display a summary table with the order count and percentage for each status, including missing values. Use all rows in `orders` as the denominator, and keep every order for later analysis.

In the final analysis, I used `order_delivered_carrier_date` as the **carrier handover timestamp**.

### A.2 Order–seller table and merge

> Firstly, copy `order_id`, `seller_id`, and `shipping_limit_date` from `order_items`. Preserve the original DataFrame. Then treat both ID columns as strings, trim whitespace, and check for missing or blank IDs. Report any affected rows before grouping. Finally, convert `shipping_limit_date` to a datetime column and report how many values were originally missing and how many failed to parse.

> Please write code to check whether any `order_id` and `seller_id` pair has conflicting shipping deadlines. If any conflicts exist, display the affected pairs and stop so I can review them. Otherwise, collapse repeated item rows to one row per `order_id`and `seller_id`, preserving the available deadline.

> Add `seller_count` for the number of distinct sellers per order and `is_multi_seller_order` to flag orders with more than one seller. Keep those orders available for later analysis.

> Save the result as `seller_orders`, verify that `order_id` and `seller_id` are unique as a pair, and print the original item row count, final seller-order row count, and number of distinct multi-seller orders.

> Write the code using pandas. I already have `orders`, with parsed purchase and carrier handover timestamps, and `seller_orders`, with one row per `order_id` and `seller_id`, a parsed `shipping_limit_date`, and an `is_multi_seller_order` flag. Please left join `seller_orders` to `orders` on `order_id`. Validate the merge as many-to-one. Report the row count before and after the merge and the number of unmatched rows. Verify that each `order_id` and `seller_id` pair remains unique.

> could u rename the merge dataset as `merged_orders`

### A.3 Late handover eligibility and seller summary

> write a code to add flags for missing handover timestamps and for handover timestamps earlier than the purchase timestamp. Preserve the original timestamps. Then Create `analysis_orders` from single-seller orders and add an `eligible_for_late_rate` flag that requires a valid purchase timestamp, carrier handover timestamp, and shipping deadline, with handover at or after purchase. Create `late_handover` as a nullable Boolean. Finally, set it to `True` when an eligible order was handed over after its shipping deadline, `False` when it was handed over on or before the deadline, and `pd.NA` when eligibility requirements are not met.

> It has bug. I write a code to add before your code. timestamp_columns = [
>     "order_purchase_timestamp",
>     "order_delivered_carrier_date",
>     "shipping_limit_date",
> ]
>
> for column in timestamp_columns:
>     merged_orders[column] = pd.to_datetime(
>         merged_orders[column], errors="raise"
>     )

> Write the next Jupyter Notebook cell using pandas. I already have `analysis_orders`, with one row per single-seller order and the columns `seller_id`, `order_id`, `eligible_for_late_rate`, and `late_handover`. The nullable Boolean column `late_handover` is `True` for late, `False` for on time, and `pd.NA` for unknown.

> Group by `seller_id` and save the result as `seller_summary`, with one row per seller. Calculate `order_volume` from all rows, `eligible_orders` from rows eligible for the late rate, `late_orders` from rows marked `True`, and `late_rate` as `late_orders / eligible_orders`. Leave `late_rate` missing when `eligible_orders` is zero.

> Verify that the totals across sellers and Display the top 10 sellers by late order count

### A.4 Monthly benchmarks and seller comparisons

> I already have `analysis_orders`, with one row per single-seller order and the columns `seller_id`, `order_purchase_month`, `eligible_for_late_rate`, and `late_handover`. please calculate the overall and monthly late handover rates across the single-seller analysis sample. For each month, divide the total number of late orders by the total number of eligible orders in that month. Then calculate the same counts and rates by seller and month, and merge the monthly benchmark onto the seller-month table. Save the result as `seller_monthly_summary`.

In my next message, I shared the revised notebook cell and asked ChatGPT to use it going forward. The excerpt below shows the change I made before grouping by month.

> It has bug. so I rewrite the code. Now it becomes
>
> ```python
> analysis_orders["order_purchase_month"] = pd.to_datetime(
>     analysis_orders["order_purchase_timestamp"],
>     errors="raise",
> ).dt.to_period("M")
> ```
>
> […] please remember it

> Then add a column for the seller's late rate minus the monthly benchmark, expressed in percentage points. Leave rates missing when the eligible order count is zero. Keep the eligible order count visible so I can interpret results based on small samples. Display the monthly benchmark table and a preview of `seller_monthly_summary`.

### A.5 Data review and rebuild with Claude

- **Review.** I had Claude review our project proposal slides and all of our CSV files, and based on those, check the data cleaning and merging code I had written earlier. Claude pointed out several issues: missing handover timestamps; undelivered orders being included in the late handover rate; out-of-order timestamps; shipping deadlines in 2020; too little data for 2016 and for September–October 2018; and a total sample size that did not match the 93,859 orders in the proposal.
- **Decision-making.** For each issue, Claude provided recommendations, but I made the final decision. I decided to exclude undelivered orders and orders with out-of-order timestamps (without filling in missing values); keep the group B orders for a credibility check; exclude orders with deadlines in 2020; exclude 2016 data; define late handover by date; and leave the minimum-order threshold to the clustering team. Every decision, its rationale and the code used to identify each issue are documented in `notebooks/change.markdown`.
- **Rebuild.** I used Claude to rewrite the cleaning notebook based on `change.markdown`. I also used Claude to change how the data is loaded, so the raw data now loads from our GitHub repository and the cleaned files are saved to the `Data/Clean/` directory.